# **1. Import Lib**

In [1]:
# Cài đặt thư viện nếu chưa có
!pip install gensim tensorflow

import pandas as pd
import numpy as np
import os
import pickle
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

# Thư viện Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, Bidirectional

# Kết nối với Google Drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 21.2 MB/s eta 0:00:00
Mounted at /content/drive


# **2. Prepare Data**

In [2]:
path_project = "/content/drive/MyDrive/FakeNewsDetection_Project"
path_dataset = os.path.join(path_project, "Dataset")

# Nạp dữ liệu từ thư mục Dataset
df_true = pd.read_csv(os.path.join(path_dataset, "True.csv"))
df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))

df_true['label'] = 1
df_fake['label'] = 0

df = pd.concat([df_true, df_fake], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df = df[['text', 'label']].dropna()

# Sử dụng cùng kích thước mẫu (ví dụ: 5000) để đảm bảo tính công bằng khi so sánh với RNN
df_sub = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(f"Dữ liệu sẵn sàng cho LSTM: {len(df_sub)} mẫu.")

/tmp/ipykernel_803/3176312996.py:6: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fake = pd.read_csv(os.path.join(path_dataset, "Fake.csv"))


Dữ liệu sẵn sàng cho LSTM: 5000 mẫu.


# **3. Prepare Training Data & Word2Vec Model**

In [3]:
df_sub['tokenized_text'] = df_sub['text'].apply(lambda x: simple_preprocess(str(x)))

X_train_words, X_test_words, y_train, y_test = train_test_split(
    df_sub['tokenized_text'], df_sub['label'], test_size=0.2, random_state=42
)

# Huấn luyện Word2Vec (100 chiều)
w2v_model = Word2Vec(sentences=X_train_words, vector_size=100, window=5, min_count=2, workers=4)

# Cấu hình chuỗi số với Keras
max_words = 20000
max_len = 200

keras_tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
keras_tokenizer.fit_on_texts(X_train_words)

X_train_seq = keras_tokenizer.texts_to_sequences(X_train_words)
X_test_seq = keras_tokenizer.texts_to_sequences(X_test_words)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

# Xây dựng Embedding Matrix từ Word2Vec
word_index = keras_tokenizer.word_index
num_words_needed = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words_needed, 100))

for word, i in word_index.items():
    if i < max_words:
        if word in w2v_model.wv:
            embedding_matrix[i] = w2v_model.wv[word]

print("Đã chuẩn bị xong lớp nhúng từ Word2Vec cho mạng LSTM!")

Đã chuẩn bị xong lớp nhúng từ Word2Vec cho mạng LSTM!


# **4. Training with LSTM**

In [4]:
# Thiết lập kiến trúc mạng LSTM
model = Sequential([
    # Lớp Embedding nhận trọng số từ Word2Vec tĩnh (trainable=False)
    Embedding(num_words_needed, 100, weights=[embedding_matrix], input_length=max_len, trainable=False),

    # Lớp LSTM với 64 bộ nhớ (units) kèm cơ chế chống quá khớp trực tiếp bên trong lớp
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),

    # Lớp Dropout bổ sung
    Dropout(0.3),

    # Lớp đầu ra phân loại nhị phân
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

print("\nĐang bắt đầu huấn luyện mạng LSTM trên GPU T4...")
history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)


Đang bắt đầu huấn luyện mạng LSTM trên GPU T4...
Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 23s 280ms/step - accuracy: 0.6855 - loss: 0.5797 - val_accuracy: 0.8290 - val_loss: 0.4371
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 259ms/step - accuracy: 0.8565 - loss: 0.3937 - val_accuracy: 0.8680 - val_loss: 0.4040
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 20s 254ms/step - accuracy: 0.8982 - loss: 0.3220 - val_accuracy: 0.9240 - val_loss: 0.2609
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 22s 273ms/step - accuracy: 0.9013 - loss: 0.2999 - val_accuracy: 0.9310 - val_loss: 0.2432
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 256ms/step - accuracy: 0.7947 - loss: 0.4951 - val_accuracy: 0.7580 - val_loss: 0.5042
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 18s 287ms/step - accuracy: 0.8435 - loss: 0.3707 - val_accuracy: 0.9080 - val_loss: 0.2695
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 19s 263ms/step - accuracy: 0.8907 - loss: 0.2960 - val_accuracy: 0.9030 - val_loss: 0.2700
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 16s 260ms

# **5. Đánh giá kết quả (APRF Tiêu chuẩn)**

In [5]:
# Dự đoán và làm tròn xác suất thành nhãn 0/1
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype("int32").flatten()

metrics = {
    'acc': accuracy_score(y_test, y_pred),
    'pre': precision_score(y_test, y_pred),
    'rec': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred)
}

print("\n" + "="*40)
print("KẾT QUẢ THỰC NGHIỆM: WORD2VEC + LSTM")
print("="*40)
print(f"1. Accuracy  (A): {metrics['acc']:.4f}")
print(f"2. Precision (P): {metrics['pre']:.4f}")
print(f"3. Recall    (R): {metrics['rec']:.4f}")
print(f"4. F1-Score  (F): {metrics['f1']:.4f}")
print("="*40)

32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step

KẾT QUẢ THỰC NGHIỆM: WORD2VEC + LSTM
1. Accuracy  (A): 0.9130
2. Precision (P): 0.8597
3. Recall    (R): 0.9793
4. F1-Score  (F): 0.9156


# **6. Test Sentence**

In [ ]:
path_models = os.path.join(path_project, "Models")

# 1. Tự động sinh file mô hình LSTM (.h5) vĩnh viễn trên Drive
model.save(os.path.join(path_models, "w2v_lstm_model.h5"))

# 2. Cập nhật kết quả vào file Metadata chung của toàn bộ bài tiểu luận
metadata_path = os.path.join(path_models, "model_info.txt")
with open(metadata_path, "a", encoding="utf-8") as f:
    f.write(f"- w2v_lstm_model.h5: Accuracy {metrics['acc']:.4f}, dùng Word2Vec 100D + LSTM.\n")

print("Đã tạo file model LSTM và cập nhật thông tin thành công!")

# 3. Hàm kiểm thử dự đoán một câu ngẫu nhiên ngoài tập dữ liệu
def predict_news_w2v_lstm(sentence):
    tokens = simple_preprocess(sentence)
    seq = keras_tokenizer.texts_to_sequences([tokens])
    pad = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    prob = model.predict(pad)[0][0]
    return f"TIN THẬT ({prob*100:.2f}%)" if prob > 0.5 else f"TIN GIẢ ({(1-prob)*100:.2f}%)"

sample_sentence = "The country signed a new trade agreement to boost agricultural exports next year."
print(f"\nCâu test: '{sample_sentence}'")
print(f"Mô hình Word2Vec + LSTM dự đoán: {predict_news_w2v_lstm(sample_sentence)}")